# Notebook 02: CBI Score Calculation
**Climate Burden Index** | github.com/meyeringn/climate-burden-index

This notebook computes the final Climate Burden Index (CBI) score for each U.S. Census tract using the merged dataset from Notebook 01.

**What happens here:**
1. Apply equity-adjusted weights to each variable
2. Compute raw composite score
3. Rescale to final 0–100 national percentile
4. Assign tier labels
5. Export final scored dataset

---
See `methodology/METHODOLOGY.md` for the full rationale behind weight decisions.

In [ ]:
import pandas as pd
import numpy as np

PROCESSED_DIR = '../data/processed/'
master = pd.read_csv(PROCESSED_DIR + 'cbi_master_dataset.csv', dtype={'FIPS': str})

print(f'Loaded {len(master):,} tracts')
master.head(3)

In [ ]:
# -------------------------------------------------------------------
# WEIGHTS — equity-adjusted, sum to 1.0
# See METHODOLOGY.md Section: 'Equity-Adjusted Weighting' for rationale
# -------------------------------------------------------------------
WEIGHTS = {
    'svi_pctile':          0.25,  # Social vulnerability — highest weight because adaptive capacity
                                   # determines outcomes more than exposure alone
    'heat_pctile':         0.22,  # Heat — deadliest U.S. climate hazard; Disabled & elderly most at risk
    'flood_pctile':        0.20,  # Flood — rapidly worsening; FEMA maps underestimate low-income risk
    'air_quality_pctile':  0.18,  # Air quality — industrial siting discrimination most documented here
    'housing_pctile':      0.15,  # Housing instability — determines recovery capacity
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, 'Weights must sum to 1.0'
print('Weights validated. Sum:', sum(WEIGHTS.values()))

In [ ]:
# -------------------------------------------------------------------
# STEP 1: Handle missing data
# Strategy: For tracts missing 1-2 variables, rescale weights of
# available variables proportionally. For tracts missing 3+ variables,
# flag as 'insufficient data' and exclude from scored output.
# -------------------------------------------------------------------

score_cols = list(WEIGHTS.keys())

# Count available variables per tract
master['vars_available'] = master[score_cols].notna().sum(axis=1)

# Tracts with fewer than 3 variables: flag and exclude from final output
insufficient = master['vars_available'] < 3
print(f'Tracts with insufficient data (< 3 variables): {insufficient.sum():,} ({insufficient.mean():.1%})')

# For remaining tracts, compute weighted score with proportional rescaling
def compute_weighted_score(row, weights):
    available = {k: v for k, v in weights.items() if pd.notna(row[k])}
    if len(available) < 3:
        return np.nan
    total_weight = sum(available.values())
    score = sum(row[k] * w / total_weight for k, w in available.items())
    return score

master['cbi_raw'] = master.apply(compute_weighted_score, axis=1, weights=WEIGHTS)

print(f'\nRaw CBI score computed for {master["cbi_raw"].notna().sum():,} tracts')
print(f'Mean raw score: {master["cbi_raw"].mean():.1f}')
print(f'Std dev: {master["cbi_raw"].std():.1f}')

In [ ]:
# -------------------------------------------------------------------
# STEP 2: Rescale to final 0–100 national percentile
# This is done among scored tracts only (excludes insufficient data)
# -------------------------------------------------------------------

scored = master[master['cbi_raw'].notna()].copy()
scored['cbi_score'] = scored['cbi_raw'].rank(pct=True) * 100
scored['cbi_score'] = scored['cbi_score'].round(1)

print(f'Final CBI score distribution:')
print(scored['cbi_score'].describe())

In [ ]:
# -------------------------------------------------------------------
# STEP 3: Assign tier labels
# -------------------------------------------------------------------

def assign_tier(score):
    if score >= 80:  return '🔴 Critical'
    elif score >= 60: return '🟠 High'
    elif score >= 40: return '🟡 Elevated'
    elif score >= 20: return '🟢 Moderate'
    else:             return '⚪ Lower'

scored['cbi_tier'] = scored['cbi_score'].apply(assign_tier)

tier_counts = scored['cbi_tier'].value_counts()
print('Tier distribution:')
print(tier_counts)
print(f'\nBy design, each tier should contain ~20% of scored tracts.')
print('Small deviations are expected due to tied scores at tier boundaries.')

In [ ]:
# -------------------------------------------------------------------
# STEP 4: Identify the primary driver for each tract
# The 'driver' is the component variable with the highest individual
# percentile score — useful for plain-language explanation
# -------------------------------------------------------------------

DRIVER_LABELS = {
    'svi_pctile':         'Social vulnerability',
    'heat_pctile':        'Heat exposure',
    'flood_pctile':       'Flood risk',
    'air_quality_pctile': 'Air quality burden',
    'housing_pctile':     'Housing instability',
}

scored['primary_driver'] = scored[score_cols].idxmax(axis=1).map(DRIVER_LABELS)

print('Most common primary drivers:')
print(scored['primary_driver'].value_counts())

In [ ]:
# -------------------------------------------------------------------
# STEP 5: Export final scored dataset
# -------------------------------------------------------------------

output_cols = [
    'FIPS', 'STATE', 'COUNTY', 'LOCATION',
    'cbi_score', 'cbi_tier', 'primary_driver',
    'svi_pctile', 'heat_pctile', 'flood_pctile', 
    'air_quality_pctile', 'housing_pctile',
    'EP_DISABL', 'EP_POC', 'E_TOTPOP',
    'vars_available'
]

output = scored[output_cols].copy()

# Save full national dataset
output.to_csv(PROCESSED_DIR + 'cbi_scores_national.csv', index=False)
print(f'Full national scores saved: {len(output):,} tracts')

# Save summary statistics by state
state_summary = output.groupby('STATE').agg(
    mean_cbi=('cbi_score', 'mean'),
    critical_tracts=('cbi_tier', lambda x: (x == '🔴 Critical').sum()),
    total_tracts=('cbi_score', 'count')
).round(1).reset_index()
state_summary['critical_pct'] = (state_summary['critical_tracts'] / state_summary['total_tracts'] * 100).round(1)
state_summary.to_csv(PROCESSED_DIR + 'cbi_state_summary.csv', index=False)
print(f'State summary saved.')

# Preview top 10 highest-burden tracts
print('\nTop 10 highest Climate Burden Index tracts:')
output.nlargest(10, 'cbi_score')[['LOCATION', 'cbi_score', 'cbi_tier', 'primary_driver']].to_string(index=False)

---
## Next Step
Proceed to **`03_equity_validation.ipynb`** to run the racial, income, and disability disparity audits.

*Score calculation complete. Results in `/data/processed/cbi_scores_national.csv`.*